In [2]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

sys.path.append("../src/data/features")
from build_features import load_prices_long, add_log_returns, add_lagged_returns, add_rolling_volatility, add_rolling_z_score, add_rsi
sys.path.append("../src/data/targets")
from build_targets import add_volatility_target, add_direction_target


In [3]:
tickers = ['SPY','GOOG', 'NVDA', 'NFLX', 'MSFT']
df = load_prices_long("../data/raw_prices.parquet", tickers)
df = add_log_returns(df)
df = add_lagged_returns(df)
print(df.head(10))


Price       date       open       high        low      close    volume ticker  \
1648  2020-01-02  66.491263  67.809142  66.491263  67.770981  28132000   GOOG   
1649  2020-01-03  66.803994  68.025229  66.689207  67.438400  23728000   GOOG   
1650  2020-01-06  66.910074  69.214751  66.910074  69.101257  34646000   GOOG   
1651  2020-01-07  69.286129  69.536417  68.911427  69.058136  30054000   GOOG   
1652  2020-01-08  68.995682  69.962167  68.934228  69.602341  30560000   GOOG   
1653  2020-01-09  70.407750  70.742794  69.897244  70.371071  30018000   GOOG   
1654  2020-01-10  70.754176  71.119409  70.297705  70.861732  36414000   GOOG   
1655  2020-01-13  71.178936  71.396516  70.677854  71.332581  33046000   GOOG   
1656  2020-01-14  71.321681  71.459957  70.794333  70.918732  31178000   GOOG   
1657  2020-01-15  70.885513  71.439875  70.885513  71.331085  25654000   GOOG   

Price  log_return  return1d  return5d  return10d  return20d  
1648          NaN       NaN       NaN        N

In [ ]:
lags = [1, 5, 10, 20]

for t in tickers:
    # Filters row for current ticker
    sub_df = df[df['ticker'] == t].dropna()
    
    # Plot each return column using dat as index
    plt.figure(figsize=(10, 4))
    for lag in lags:
        plt.plot(sub_df['date'], sub_df[f'return{lag}d'], label=f'{lag}d Return')
        
    plt.title(f'Lagged Log Returns - {t}')
    plt.xlabel('Date')
    plt.ylabel('Log Return')
    plt.legend()
    plt.tight_layout()  
    plt.show()

In [4]:
df = add_rolling_volatility(df)

In [ ]:
windows = [5, 10, 20]

for t in tickers:
    # Filters row for current ticker
    sub_df = df[df['ticker'] == t].dropna()
    
    # Plot each return column using dat as index
    plt.figure(figsize=(10, 4))
    for v in windows:
        plt.plot(sub_df['date'], sub_df[f'vol_{v}d'], label=f'{v}d Volatility')
        
    plt.title(f'Rolling Volatility - {t}')
    plt.xlabel('Date')
    plt.ylabel('Rolling Volatility')
    plt.legend()
    plt.tight_layout()  
    plt.show()

In [5]:
df = add_rolling_z_score(df)

In [ ]:
windows = [5, 10, 20]

for t in tickers:
    # Filters row for current ticker
    sub_df = df[df['ticker'] == t].dropna()
    
    # Plot each return column using dat as index
    plt.figure(figsize=(10, 4))
    for z in windows:
        plt.plot(sub_df['date'], sub_df[f'z_{v}d'], label=f'{v}d Z Score')
        
    plt.title(f'Rolling Z Score - {t}')
    plt.xlabel('Date')
    plt.ylabel('Rolling Z Score')
    plt.legend()
    plt.tight_layout()  
    plt.show()

In [6]:
df = add_rsi(df)

In [ ]:
windows = [9, 7, 14, 25]

for t in tickers:
    # Filters row for current ticker
    sub_df = df[df['ticker'] == t].dropna()
    
    # Plot each return column using date as index
    plt.figure(figsize=(10, 4))
    for r in windows:
        plt.plot(sub_df['date'], sub_df[f'rsi_{r}d'], label=f'{r}d RSI')
        
    plt.title(f'RSI - {t}')
    plt.xlabel('Date')
    plt.ylabel('RSI')
    plt.legend()
    plt.tight_layout()  
    plt.show()

In [7]:
df = add_volatility_target(df)
df = add_direction_target(df)

In [11]:
df = df.sort_values(by=['ticker', 'date'])
df.head(10)

Price,date,open,high,low,close,volume,ticker,log_return,return1d,return5d,...,vol_20d,z_10d,z_20d,z_60d,rsi_9d,rsi_7d,rsi_14d,rsi_25d,target_vol_5d,target_dir_1d
1648,2020-01-02,66.491263,67.809142,66.491263,67.770981,28132000,GOOG,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.180423,0.0
1649,2020-01-03,66.803994,68.025229,66.689207,67.438400,23728000,GOOG,-0.004919,-0.004919,NaN,...,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.145036,1.0
1650,2020-01-06,66.910074,69.214751,66.910074,69.101257,34646000,GOOG,0.024358,0.024358,NaN,...,NaN,NaN,NaN,NaN,84.905308,85.365510,84.336986,83.892245,0.067721,0.0
1651,2020-01-07,69.286129,69.536417,68.911427,69.058136,30054000,GOOG,-0.000624,-0.000624,NaN,...,NaN,NaN,NaN,NaN,82.853044,83.216319,82.396332,82.033252,0.102540,1.0
1652,2020-01-08,68.995682,69.962167,68.934228,69.602341,30560000,GOOG,0.007849,0.007849,NaN,...,NaN,NaN,NaN,NaN,87.234051,87.755294,86.590143,86.086414,0.100359,1.0
1653,2020-01-09,70.407750,70.742794,69.897244,70.371071,30018000,GOOG,0.010984,0.010984,0.037648,...,NaN,NaN,NaN,NaN,90.920529,91.530176,90.157263,89.553926,0.092510,1.0
1654,2020-01-10,70.754176,71.119409,70.297705,70.861732,36414000,GOOG,0.006948,0.006948,0.049516,...,NaN,NaN,NaN,NaN,92.479881,93.111541,91.678771,91.038777,0.143449,1.0
1655,2020-01-13,71.178936,71.396516,70.677854,71.332581,33046000,GOOG,0.006623,0.006623,0.031780,...,NaN,NaN,NaN,NaN,93.656106,94.302479,92.824981,92.153650,0.146696,0.0
1656,2020-01-14,71.321681,71.459957,70.794333,70.918732,31178000,GOOG,-0.005819,-0.005819,0.026586,...,NaN,NaN,NaN,NaN,81.111381,80.101578,82.118130,82.730165,0.116268,1.0
1657,2020-01-15,70.885513,71.439875,70.885513,71.331085,25654000,GOOG,0.005798,0.005798,0.024534,...,NaN,1.092749,NaN,NaN,83.577158,83.065925,84.087583,84.387217,0.126989,1.0
